# Adaptive Composite Kernel: Development and Validation

This notebook documents the progression from kernel-target alignment to an
end-to-end adaptive composite-kernel SVM in `svdynamics`.

The tests are organized to validate the method in stages:

1. Verify kernel-target alignment behavior on controlled examples.
2. Compare alignment across candidate kernels on nonlinear data.
3. Convert alignment scores into normalized kernel weights.
4. Fit `AdaptiveCompositeKernel` directly.
5. Verify the public `svdynamics` API.
6. Fit and evaluate an adaptive `SVDClassifier` end to end.

The current adaptive weighting approach learns a single global weight for each
candidate kernel from the training data.


## 1. Kernel-Target Alignment Sanity Checks

For binary labels encoded as `-1` and `+1`, the target kernel is

\[
K_y = y y^\top.
\]

Kernel-target alignment measures the similarity between a candidate kernel
matrix \(K\) and the target kernel:

\[
A(K, K_y) =
\frac{\langle K, K_y\rangle_F}
{\|K\|_F \|K_y\|_F}.
\]

The first three tests check expected edge cases:

- perfect agreement should produce alignment `1.0`,
- exact opposition should produce alignment `-1.0`,
- a zero kernel should safely return `0.0`.


In [ ]:
import numpy as np

from svdynamics.kernels import (
    _alignment_weights,
    _kernel_target_alignment,
)


y = np.array([-1, -1, 1, 1])
target_kernel = np.outer(y, y)

perfect_alignment = _kernel_target_alignment(
    target_kernel,
    y,
)

print("Target kernel:")
print(target_kernel)
print("\nPerfect alignment:", perfect_alignment)


In [ ]:
opposite_kernel = -target_kernel

opposite_alignment = _kernel_target_alignment(
    opposite_kernel,
    y,
)

print("Opposite alignment:", opposite_alignment)


In [ ]:
zero_kernel = np.zeros((4, 4))

zero_alignment = _kernel_target_alignment(
    zero_kernel,
    y,
)

print("Zero-kernel alignment:", zero_alignment)


## 2. Alignment Across Candidate Kernels

Next, use a nonlinear concentric-circles classification problem. A linear
kernel should have very little alignment with the class structure, while an
RBF kernel should be much better suited to the geometry.

The labels are converted from scikit-learn's `0/1` representation to `-1/+1`
for the current alignment implementation.


In [ ]:
from sklearn.datasets import make_circles

from svdynamics.kernels import (
    _compute_kernel,
    _parse_kernel_spec,
)


X, y = make_circles(
    n_samples=200,
    factor=0.4,
    noise=0.05,
    random_state=42,
)

y = 2 * y - 1

kernel_specs = [
    "linear",
    ("rbf", {"gamma": 1.0}),
    ("poly", {"degree": 2}),
]

kernel_matrices = []

for spec in kernel_specs:
    parsed = _parse_kernel_spec(spec)
    K = _compute_kernel(
        parsed,
        X,
        X,
    )
    kernel_matrices.append(K)

    alignment = _kernel_target_alignment(
        K,
        y,
    )
    print(f"{spec}: {alignment:.6f}")


## 3. Convert Alignment Scores to Kernel Weights

Positive alignment scores are normalized so that the resulting kernel weights
sum to one:

\[
w_m =
\frac{a_m}
{\sum_j a_j}.
\]

Negative alignment values are clipped to zero before normalization. If all
usable alignments are effectively zero, the implementation falls back to
uniform weights.


In [ ]:
weights = _alignment_weights(
    kernel_matrices,
    y,
)

print("Learned weights:")
print(weights)

print("\nWeight sum:")
print(weights.sum())


## 4. Fit `AdaptiveCompositeKernel`

The previous sections tested the underlying functions directly. Now fit the
public adaptive-kernel object and verify that it stores both the raw alignment
scores and the learned normalized weights.


In [ ]:
from svdynamics.kernels import AdaptiveCompositeKernel


adaptive_kernel = AdaptiveCompositeKernel(
    kernels=kernel_specs,
)

adaptive_kernel.fit(X, y)

print("Alignments:")
print(adaptive_kernel.alignments_)

print("\nLearned weights:")
print(adaptive_kernel.weights_)

print("\nNormalized weights:")
print(adaptive_kernel.normalized_weights)


## 5. Public API Import Check

Confirm that the adaptive kernel and existing estimators are exposed through
the top-level `svdynamics` package.


In [ ]:
from svdynamics import (
    AdaptiveCompositeKernel,
    CompositeKernel,
    SVDClassifier,
    SVDRegressor,
)


print(AdaptiveCompositeKernel)
print(CompositeKernel)
print(SVDClassifier)
print(SVDRegressor)


## 6. End-to-End Adaptive SVM Classification

Finally, split the circles data into training and test sets. The adaptive
kernel learns its alignment scores and weights from the training data when
`SVDClassifier.fit()` is called.

This test checks the complete progression:

\[
(X_{train}, y_{train})
\rightarrow
\text{kernel alignments}
\rightarrow
\text{learned weights}
\rightarrow
\text{composite kernel}
\rightarrow
\text{SVC}
\rightarrow
\hat{y}_{test}.
\]


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

kernel = AdaptiveCompositeKernel(
    kernels=kernel_specs,
)

model = SVDClassifier(
    kernel=kernel,
)

model.fit(
    X_train,
    y_train,
)

y_pred = model.predict(X_test)

print("Alignments:")
print(kernel.alignments_)

print("\nLearned weights:")
print(kernel.weights_)

print("\nWeight sum:")
print(kernel.weights_.sum())

print("\nTest accuracy:")
print(accuracy_score(y_test, y_pred))


## Current Milestone

At this stage, `svdynamics` can:

- compute kernel-target alignment,
- derive normalized global kernel weights from those alignments,
- store fitted alignment diagnostics,
- expose `AdaptiveCompositeKernel` through the package API,
- automatically fit the adaptive kernel through `SVDClassifier`,
- make end-to-end predictions using the learned composite kernel.

This notebook is a development and validation record for the current
alignment-based adaptive composite-kernel implementation.
